# 👁️ Mojo Vision Service

This notebook gives Mojo the ability to see WebGL content by running headless Chrome with GPU acceleration.

**Setup:**
1. Runtime > Change runtime type > T4 GPU
2. Run all cells
3. Copy the ngrok URL to share with Mojo

In [ ]:
#@title 1. Install Dependencies
!apt-get update -qq
!apt-get install -y vulkan-tools libnvidia-gl-525 -qq
!pip install flask pyngrok pyppeteer nest_asyncio -q
!pyppeteer-install

In [ ]:
#@title 2. Verify GPU
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
#@title 3. Configure ngrok
NGROK_TOKEN = "38mc6FxzmwWOLjcBBGNprJ9BYTo_29GQfghrZdbDx94fLQgri"  #@param {type:"string"}

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok configured!")

In [ ]:
#@title 4. Start Vision Service
import asyncio
import base64
import json
from flask import Flask, request, jsonify
from pyppeteer import launch
from pyngrok import ngrok
import nest_asyncio
nest_asyncio.apply()

app = Flask(__name__)

# Chrome launch args for GPU WebGL support
CHROME_ARGS = [
    '--no-sandbox',
    '--disable-setuid-sandbox',
    '--headless=new',
    '--use-angle=vulkan',
    '--enable-features=Vulkan',
    '--disable-vulkan-surface',
    '--enable-unsafe-webgpu',
    '--disable-dev-shm-usage',
    '--window-size=1280,720'
]

async def take_screenshot(url, wait_ms=3000):
    """Take a screenshot of a URL with WebGL support"""
    browser = await launch(
        headless=True,
        args=CHROME_ARGS
    )
    page = await browser.newPage()
    await page.setViewport({'width': 1280, 'height': 720})
    
    # Collect console messages
    console_logs = []
    page.on('console', lambda msg: console_logs.append({
        'type': msg.type,
        'text': msg.text
    }))
    
    await page.goto(url, {'waitUntil': 'networkidle0', 'timeout': 30000})
    
    # Wait for WebGL to render
    await asyncio.sleep(wait_ms / 1000)
    
    # Take screenshot
    screenshot = await page.screenshot({'encoding': 'base64'})
    
    await browser.close()
    
    return {
        'image': screenshot,
        'console': console_logs[-20:],  # Last 20 messages
        'url': url
    }

@app.route('/health')
def health():
    import subprocess
    gpu = subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader')
    return jsonify({'status': 'ok', 'gpu': gpu.strip()})

@app.route('/screenshot')
def screenshot():
    url = request.args.get('url')
    wait_ms = int(request.args.get('wait', 3000))
    
    if not url:
        return jsonify({'error': 'url parameter required'}), 400
    
    try:
        loop = asyncio.get_event_loop()
        result = loop.run_until_complete(take_screenshot(url, wait_ms))
        return jsonify(result)
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\n🔗 Vision Service URL: {public_url}")
print(f"\n📸 Screenshot endpoint: {public_url}/screenshot?url=YOUR_URL")
print(f"❤️ Health check: {public_url}/health")
print("\n⚡ Service is running! Keep this cell active.")

# Run Flask (blocking)
app.run(port=5000)